<a href="https://colab.research.google.com/github/takao-takenouchi/k-anonymization_sql/blob/main/k_anonymization_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 初めに
このノートブックでは、
個人情報データベースのサンプルデータを用いて、「k-匿名化」をSQLで行います。
「k-匿名化」の処理の流れを、簡単に理解することを目的にしています。

## 匿名化処理をSQLで体験

### Step1 テーブルの準備

In [ ]:
# DuckDBを使ってSQLで処理します
!pip -q install duckdb

サンプル用のファイルをダウンロードして使います。（ファイルをアップロードすることも可能です）

In [ ]:
import urllib.request

url = "https://raw.githubusercontent.com/takao-takenouchi/k-anonymization_sql/main/anonymization_table.csv"
urllib.request.urlretrieve(url, "anonymization_table.csv")

## ファイルをアップロードしたい場合は、以下のコードをつかいます。「ファイル選択」をクリックして、「anonymization_table.csv」をアップロードしてください。
# from google.colab import files
# uploaded = files.upload()

アップロードしたCSVファイルを、Databaseに読み込んで、SQLのテーブル作成します

In [ ]:
import duckdb

con = duckdb.connect()

# 文字列型(VARVHAR型)で読み込みます。ファイル名は先ほどアップロードした"anonymization_table.csv"です
con.execute("""
CREATE OR REPLACE TABLE anonymization AS
SELECT
    CAST(column0 AS VARCHAR) AS gender,
    CAST(column1 AS VARCHAR) AS post,
    CAST(column2 AS VARCHAR) AS age,
    CAST(column3 AS VARCHAR) AS disease
FROM read_csv_auto(
    'anonymization_table.csv',
    header = false
);
""")



# 一応テーブルの詳細を表示します
con.execute("DESCRIBE anonymization").df()

In [ ]:
# 作成したテーブルも表示します。5000行のファイルのはずです。
con.execute("SELECT * FROM anonymization").df()

### Step2 匿名化前のkの値をチェック



まずは、匿名化前のテーブルのkの値をチェックします。
準識別子でGroup byしてCountを取っていくだけです。簡単です。

In [ ]:
con.execute("""
SELECT
    k,
    COUNT(*) AS group_count
FROM (
    SELECT gender, post, age, COUNT(*) AS k
    FROM anonymization
    GROUP BY gender, post, age
) t
GROUP BY k
ORDER BY k
""").df()

### Step3 匿名化処理


続いて匿名化します。ここでは、Global Recodingで、一部の桁を削除する方法で行います。

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE anon_1 AS
SELECT
    gender,
    SUBSTRING(post, 1, 1) || '**-****' AS post,
    SUBSTRING(age, 1, 1) || '*' AS age,
    disease
FROM anonymization
""")

con.execute("SELECT * FROM anon_1").df()

### Step4 匿名化後のkの値をチェック

匿名化後のテーブルのkの値をチェックします。同じく、Group byしてCountを取っていくだけです。

In [ ]:
con.execute("""
SELECT
    k,
    COUNT(*) AS group_count
FROM (
    SELECT gender, post, age, COUNT(*) AS k
    FROM anon_1
    GROUP BY gender, post, age
) t
GROUP BY k
ORDER BY k
""").df()

## まとめ

- SQLで匿名化処理の概要を体験
- Global Recodingで桁の削除であれば、SQL一発
- 匿名化前と匿名化後のkの値を比較も簡単。kの値のチェックはGroup byしてCountを取っていくだけ
